# ingest large parquet

## 5.1. read path

In [0]:
ROOT_PATH = "/Volumes/workspace/bronze/logistic_landing"

display(dbutils.fs.ls(ROOT_PATH))

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/bronze/logistic_landing/batch_large"
    )
)

In [0]:
BASE_PATH = (
    "/Volumes/workspace/bronze/logistic_landing/batch_large"
)

PARQUET_PATH = f"{BASE_PATH}/shipments_100k.parquet"


print("Parquet files:")
display(dbutils.fs.ls(f"{PARQUET_PATH}"))

## 5.2. Check number of rows

In [0]:
parquet_count = (
    spark.read
    .parquet(PARQUET_PATH)
    .count()
)

print(f"Parquet rows: {parquet_count:,}")

assert parquet_count == 100_000

## 5.3 Ingest large parquet file into a table


In [0]:

import time

TARGET_TABLE = "workspace.bronze.shipments_large_parquet"

start = time.perf_counter()

parquet_df = spark.read.parquet(PARQUET_PATH)

(
    parquet_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

elapsed = time.perf_counter() - start

print(f"Parquet ingestion time: {elapsed:.2f} seconds")
print(f"Target: {TARGET_TABLE}")
